[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# SQL Syntax


## What you will be able to do

Write a `SELECT` with its clauses in the order SQL requires, and predict its result from the order
in which those clauses take effect. Summarize rows with `COUNT`, `SUM`, `AVG`, `MIN` and `MAX`, for
a whole table or for groups, and keep only the groups you want with `HAVING`. Ask questions about
dates and times in a `WHERE` clause with ranges of text, SQLite's date and time functions and Unix
timestamps, and tell the kinds of SQL statement apart, including the ones SQLite does not have.


## The idea

### SQL at a glance

A summary to come back to, before any of the detail.

#### The SELECT statement

A query's clauses are written in one order, and take effect in another. The written order is the
one SQL requires. The logical order is the one that decides the result:

| Clause | What it does | Written order | Logical order |
|---|---|---|---|
| `FROM` and `JOIN` | names the tables, and matches up their rows | 2 | 1 |
| `WHERE` | keeps the rows a condition is true for, before any grouping | 3 | 2 |
| `GROUP BY` | collects rows into groups, one result row for every group | 4 | 3 |
| `HAVING` | keeps the groups a condition is true for, after aggregating | 5 | 4 |
| `SELECT` | works out the columns to return, and `DISTINCT` removes repeated rows | 1 | 5 |
| `ORDER BY` | sorts the result, ascending unless a column says `DESC` | 6 | 6 |
| `LIMIT` and `OFFSET` | keeps a number of rows, after skipping some | 7 | 7 |

SQLite writes `LIMIT`, as PostgreSQL and MySQL do. SQL Server writes `TOP` instead, straight after
`SELECT`, and SQLite reads that as a mistake, which one of the Common errors shows.

#### The four sub-languages

SQL's statements fall into groups by what they do to a database:

| Group | What it does | Statements | In SQLite |
|---|---|---|---|
| DQL, data query language | reads data | `SELECT` | as written here |
| DML, data manipulation language | changes the rows in tables | `INSERT INTO t (columns) VALUES (values)`, `UPDATE t SET column = value WHERE condition`, `DELETE FROM t WHERE condition` | as written here |
| DDL, data definition language | creates, changes and removes structures | `CREATE TABLE`, `ALTER TABLE t ADD COLUMN`, `DROP TABLE`, `TRUNCATE TABLE` | no `TRUNCATE`: `DELETE FROM t` with no `WHERE` empties a table and keeps it |
| DCL, data control language | grants and removes permissions | `GRANT`, `REVOKE` | neither: SQLite has no users, and the file's own permissions decide who can read and write it |

Some references count `SELECT` as DML, and many add a fifth group, TCL, transaction control language,
for `BEGIN`, `COMMIT`, `ROLLBACK` and `SAVEPOINT`, which the **Transactions** notebook covers.

#### Syntax rules and conventions

- **Case.** Keywords are case-insensitive, so `select` and `SELECT` are the same keyword, and SQLite
  matches table and column names without regard to case too. Keywords in capitals are a widespread
  convention, followed throughout this guide, that sets them apart from names. Text in single quotes
  keeps its case, and `=` compares it exactly.
- **Semicolons.** A semicolon ends a statement. sqlite3's `execute` runs a single statement, with or
  without one, and `executescript`, which runs several, needs one between every two statements.
- **Comments.** `--` starts a comment that runs to the end of the line, and `/* ... */` encloses one
  that can span several lines or sit in the middle of one.
- **Quotes.** Text goes in single quotes, and a name may go in double quotes, as the **Tables and
  Queries** notebook showed.
- **Layout.** Line breaks and indentation mean nothing to SQL, so a query can be laid out for
  reading.

### The problem

The **Tables and Queries** notebook wrote a query of every common kind once, enough to use each. The
questions people actually ask combine them, and the combinations are where SQL surprises. What share
of each station's readings were below freezing? That takes groups, an aggregate over a condition, and
arithmetic that does not quietly throw the fraction away. Which days in March stayed below -6 degrees
all day? That keeps some groups and not others by their warmest reading, which `WHERE`, looking at
one reading at a time, cannot do. Which readings fall in March? That compares dates stored as text.

Each of those can be written so that it runs without an error and returns the wrong answer: a date
range that leaves out the last day of the month, an `OR` that binds more loosely than it looks, a
percentage that comes out as 0. A few rules answer all of them in advance: the order the clauses take
effect in, how `AND`, `OR` and division work, and what SQLite's date and time functions read and
return.

### What SQL is

> **SQL**, the structured query language, is the language relational databases share for defining,
> changing and asking questions of data. A **statement** is one complete instruction, and a
> **query** is a `SELECT` statement, made of **clauses**: `SELECT`, `FROM`, `WHERE`, `GROUP BY`,
> `HAVING`, `ORDER BY` and `LIMIT`, always written in that order, and all optional except `SELECT`
> itself. An **aggregate function**, such as `COUNT`, `SUM`, `AVG`, `MIN` or `MAX`, turns a set of
> rows into a single value, for the whole result or for every group that `GROUP BY` makes. SQLite
> has no date or time type: a time is stored as ISO 8601 text, a Unix timestamp or a Julian day
> number, and its **date and time functions**, `date`, `time`, `datetime`, `julianday` and
> `strftime`, read and write all three.

### Why it works that way

- **The written order serves the reader, and the logical order defines the result.** `FROM` takes
  effect first and `SELECT` fifth, which is why, in standard SQL, a name given with `AS` in `SELECT`
  cannot be used in `WHERE`. SQLite accepts it anyway, and PostgreSQL does not, so SQL meant to run
  anywhere repeats the expression.
- **The logical order is a meaning, not a plan.** SQLite may do the work in any order that gives the
  same result, and use an index to skip most of the table named in `FROM`, as the **Indexes and Query
  Plans** notebook shows.
- **Aggregates skip `NULL`.** `COUNT(column)`, `SUM`, `AVG`, `MIN` and `MAX` ignore missing values,
  while `COUNT(*)` counts rows. `SUM` of no values at all is `NULL`, where SQLite's `TOTAL` is 0.0.
- **Dividing two integers gives an integer.** `7 / 2` is 3 in SQLite, so a share worked out from two
  counts needs a real number in it, such as `100.0`.
- **`AND` binds more tightly than `OR`.** `a OR b AND c` means `a OR (b AND c)`, so a condition that
  mixes the two needs parentheses to say anything else.
- **ISO 8601 text sorts in time order.** A range of such text in `WHERE` finds the times in a span,
  and the functions work out what text alone cannot say: a weekday, a month's last day, or the hours
  between two times.

### Where this shows up

Most of this notebook's SQL is shared by every relational database, and where SQLite differs, the
notebook says so. PostgreSQL, in the **asyncpg and psycopg3, Deep Dive** guide, has `TRUNCATE`,
`GRANT` and `REVOKE`, and stores dates and timestamps in types of their own, which it takes apart
with `EXTRACT` and `date_trunc` where SQLite uses `strftime`. The **DuckDB, Deep Dive** guide writes
the same clauses over CSV and Parquet files, and SQL Server writes `TOP` where SQLite writes
`LIMIT`. In Python, the **SQLAlchemy, Deep Dive** guide builds a query with methods named after its
clauses, such as `where` and `group_by`, and the **Pandas, Deep Dive** guide reads the parts of a
time with a column's `dt` accessor. The **Joins** notebook takes `FROM` and `JOIN` apart.

### What this notebook covers

- The clauses of a query, run one at a time in the order they take effect
- `SELECT`: expressions, `AS`, `CASE`, `DISTINCT` and a few functions
- `WHERE`: comparisons, `AND`, `OR` and `NOT`, `IN`, `BETWEEN`, `LIKE`, `GLOB` and `IS NULL`
- `ORDER BY` with `NULLS FIRST` and `NULLS LAST`, and `LIMIT` with `OFFSET`
- Aggregate functions, with `GROUP BY`, `HAVING` and `FILTER`
- Dates, times and timestamps in `WHERE`: ranges, `date`, `time`, `datetime`, `strftime`,
  `julianday`, modifiers, and Unix timestamps
- Statements that change data and structure: `INSERT`, `UPDATE`, `DELETE`, `CREATE`, `ALTER` and
  `DROP`
- Comments, semicolons and case
- When to use a range, `date`, `strftime` or `BETWEEN` for a time in `WHERE`
- A monthly frost summary for two stations
- Seven errors: `TOP` for `LIMIT`, `WHERE` after `GROUP BY`, `TRUNCATE`, `BETWEEN` and the last day,
  `AND` beside `OR`, dividing two counts, and dates written day first

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE readings (station TEXT, hour TEXT, celsius REAL);
    INSERT INTO readings VALUES
        ('Oslo', '2025-03-01T06:00', -3.1), ('Oslo', '2025-03-01T18:00', 1.2),
        ('Bergen', '2025-03-01T06:00', 2.4), ('Bergen', '2025-03-01T18:00', 4.9),
        ('Tromso', '2025-03-01T06:00', -7.5), ('Tromso', '2025-03-02T06:00', -8.0);
""")

query = """
    SELECT station, COUNT(*) AS readings, ROUND(AVG(celsius), 2) AS mean  -- 5: the columns
    FROM readings                                                          -- 1: the table
    WHERE hour >= '2025-03-01' AND hour < '2025-03-02'                     -- 2: 1 March only
    GROUP BY station                                                       -- 3: a group per station
    HAVING COUNT(*) = 2                                                    -- 4: groups of two
    ORDER BY mean DESC                                                     -- 6: warmest first
    LIMIT 2;                                                               -- 7: two rows at most
"""
for row in conn.execute(query):
    print(row)
conn.close()
```

```
('Bergen', 2, 3.65)
('Oslo', 2, -0.95)
```

One query, with every clause of a `SELECT`, and its comments numbering the clauses in the order they
take effect. `WHERE` kept the five readings from 1 March, `GROUP BY` made a group for every station,
and `HAVING` dropped Tromso, which had only one reading that day. Then `SELECT` worked out the count
and mean of the two groups left, and `ORDER BY` and `LIMIT` put the warmer station first.


## Setup

Five imports, and the stations' year, built into the two tables the **Tables and Queries** notebook
designed.

- `sqlite3` builds the database and runs every statement
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end

`LATITUDES` has one station more than `STATIONS`: Kirkenes, which joined the network too recently to
have sent a reading.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### The clauses, in the order they take effect

A query can be taken apart in its logical order, one clause at a time, counting what is left after
each. `JOINED` is the `FROM` clause every query below starts from, and `SVALBARD_IN_MARCH` a
condition several of them share. Both are fixed text in this notebook, never input from a user, which
is what makes it safe to write them into the SQL with an f-string, as the **Parameters** notebook
explains:


In [2]:
conn = sqlite3.connect(DATABASE)
JOINED = "FROM readings AS r JOIN stations AS s ON s.id = r.station_id"
SVALBARD_IN_MARCH = "s.name = 'Svalbard' AND r.hour >= '2025-03-01' AND r.hour < '2025-04-01'"

steps = [
    ("1. FROM and JOIN: every row", f"SELECT COUNT(*) {JOINED}"),
    ("2. WHERE: Svalbard in March", f"SELECT COUNT(*) {JOINED} WHERE {SVALBARD_IN_MARCH}"),
    ("3. GROUP BY: a group for every day",
     f"SELECT COUNT(*) FROM (SELECT 1 {JOINED} WHERE {SVALBARD_IN_MARCH} GROUP BY date(r.hour))"),
    ("4. HAVING: days below -6 all day",
     f"SELECT COUNT(*) FROM (SELECT 1 {JOINED} WHERE {SVALBARD_IN_MARCH}"
     f" GROUP BY date(r.hour) HAVING MAX(r.celsius) < -6)"),
]
for label, sql in steps:
    print(f"{label:<36} {conn.execute(sql).fetchone()[0]:>6} rows")

coldest_days = f"""
    SELECT date(r.hour) AS day, MAX(r.celsius) AS warmest
    {JOINED}
    WHERE {SVALBARD_IN_MARCH}
    GROUP BY date(r.hour)
    HAVING MAX(r.celsius) < -6
    ORDER BY warmest, day
    LIMIT 3
"""
print("5 to 7. SELECT, ORDER BY and LIMIT:", conn.execute(coldest_days).fetchall())


1. FROM and JOIN: every row           35040 rows
2. WHERE: Svalbard in March             744 rows
3. GROUP BY: a group for every day       31 rows
4. HAVING: days below -6 all day         14 rows
5 to 7. SELECT, ORDER BY and LIMIT: [('2025-03-01', -8.0), ('2025-03-05', -7.7), ('2025-03-03', -7.2)]


`FROM` and `JOIN` made 35,040 rows, one for every station's hour. Of those, 744 were left once
`WHERE` kept Svalbard in March, `GROUP BY` made 31 groups, one for every day, and `HAVING` kept the
14 days whose warmest reading was below -6 degrees. A reading fell below -6 degrees on 30 of March's
days, so a `WHERE` on single readings could not have found those 14. Only then did `SELECT` work out
every day's warmest reading, `ORDER BY` sort the days, with `day` deciding between 3 and 6 March,
which tie at -7.2, and `LIMIT` keep three. The counts at steps 3 and 4 wrap the query in an outer
`SELECT COUNT(*)`, a query over the rows of another query, which counts the groups.

### SELECT: the columns a query returns

`SELECT` can return any expression, not only columns: arithmetic, text joined with `||`, functions
such as `upper`, `length`, `round` and `coalesce`, and a choice made with `CASE`. `AS` names each
result, and `DISTINCT` removes rows that repeat:


In [3]:
for row in conn.execute("""
    SELECT upper(name) AS station,
           length(name) AS letters,
           name || ' at ' || latitude || ' degrees' AS label,
           CASE WHEN latitude > 66.56 THEN 'Arctic' ELSE 'south of the Arctic Circle' END AS zone
    FROM stations
    ORDER BY latitude DESC
"""):
    print(row)

months = conn.execute("SELECT DISTINCT strftime('%Y-%m', hour) FROM readings").fetchall()
print("months with readings:", len(months))
print("a missing reading shown with coalesce:", conn.execute(f"""
    SELECT r.hour, coalesce(r.celsius, 'no reading') {JOINED} WHERE s.name = 'Svalbard' AND r.hour = '2025-03-02T06:00'
""").fetchone())


('SVALBARD', 8, 'Svalbard at 78.22 degrees', 'Arctic')
('KIRKENES', 8, 'Kirkenes at 69.73 degrees', 'Arctic')
('TROMSO', 6, 'Tromso at 69.65 degrees', 'Arctic')
('BERGEN', 6, 'Bergen at 60.39 degrees', 'south of the Arctic Circle')
('OSLO', 4, 'Oslo at 59.91 degrees', 'south of the Arctic Circle')
months with readings: 12
a missing reading shown with coalesce: ('2025-03-02T06:00', 'no reading')


`CASE` checks its `WHEN` conditions in order and returns the first match, or `ELSE`: the Arctic
Circle lies at about 66.56 degrees north. `DISTINCT` turned 35,040 months, one for each row of
readings, into 12, and `coalesce` returned its first argument that is not `NULL`, so a missing
reading came back as text.

### WHERE: conditions, and how AND and OR combine

`WHERE` keeps a row when its condition is true, and conditions combine with `AND`, `OR` and `NOT`.
`AND` binds more tightly than `OR`, so parentheses decide what goes with what. `IN` tests a list,
`BETWEEN` a range including both ends, `LIKE` a pattern in any case, where `%` stands for any text,
and `GLOB` a pattern in exact case, where `*` does:


In [4]:
def count_where(condition):
    """How many joined rows a condition keeps."""
    return conn.execute(f"SELECT COUNT(*) {JOINED} WHERE {condition}").fetchone()[0]


for condition in [
    "r.celsius < 0",
    "(s.name = 'Oslo' OR s.name = 'Bergen') AND r.celsius < 0",
    "s.name IN ('Oslo', 'Bergen') AND r.celsius < 0",
    "NOT s.name IN ('Oslo', 'Bergen') AND r.celsius BETWEEN -1 AND 1",
    "s.name LIKE 's%' AND r.celsius IS NULL",
    "s.name GLOB 's*'",
]:
    print(f"{count_where(condition):>6}  WHERE {condition}")


 12142  WHERE r.celsius < 0
  3068  WHERE (s.name = 'Oslo' OR s.name = 'Bergen') AND r.celsius < 0
  3068  WHERE s.name IN ('Oslo', 'Bergen') AND r.celsius < 0
  1627  WHERE NOT s.name IN ('Oslo', 'Bergen') AND r.celsius BETWEEN -1 AND 1
    24  WHERE s.name LIKE 's%' AND r.celsius IS NULL
     0  WHERE s.name GLOB 's*'


The second and third conditions are the same question written two ways, and both needed the stations
kept together before `AND` applied, by parentheses or by `IN`. `NOT` turned `IN` around. `LIKE 's%'`
matched Svalbard in spite of the capital, and found its 24 missing readings, while `GLOB 's*'`
matched no station, since `GLOB` compares case exactly.

### ORDER BY and LIMIT

`ORDER BY` sorts by its first expression, then by the next where the first is equal, as `day` did for
two days in March. `DESC` reverses one of them, and `NULLS FIRST` or `NULLS LAST` says where missing
values go, since SQLite otherwise sorts `NULL` below every value. `LIMIT` keeps a number of rows, and
`OFFSET` skips some first. The same four readings, from midnight on 2 March, sorted four ways:


In [5]:
at_midnight = f"SELECT s.name, r.celsius {JOINED} WHERE r.hour = '2025-03-02T00:00'"

for rest in ["r.celsius", "r.celsius NULLS LAST", "r.celsius DESC", "r.celsius DESC LIMIT 2 OFFSET 1"]:
    print(f"ORDER BY {rest:<31}", conn.execute(f"{at_midnight} ORDER BY {rest}").fetchall())


ORDER BY r.celsius                       [('Svalbard', None), ('Tromso', -4.4), ('Oslo', -2.9), ('Bergen', -1.3)]
ORDER BY r.celsius NULLS LAST            [('Tromso', -4.4), ('Oslo', -2.9), ('Bergen', -1.3), ('Svalbard', None)]
ORDER BY r.celsius DESC                  [('Bergen', -1.3), ('Oslo', -2.9), ('Tromso', -4.4), ('Svalbard', None)]
ORDER BY r.celsius DESC LIMIT 2 OFFSET 1 [('Oslo', -2.9), ('Tromso', -4.4)]


Svalbard, silent that day, came first in the ascending sort, since its missing reading sorts below
every temperature, and `NULLS LAST` moved it to the end. Sorting in descending order put it last
anyway, and `LIMIT 2 OFFSET 1` skipped the warmest and kept the next two. `NULLS FIRST` and
`NULLS LAST` need SQLite 3.30.0, from 2019.

### Aggregate functions

An aggregate function turns many rows into one value. With no `GROUP BY`, the whole result is one
group. `COUNT(*)` counts rows, and every other aggregate skips `NULL`. `COUNT(DISTINCT ...)` counts
different values, and `TOTAL` is SQLite's `SUM` that returns 0.0 rather than `NULL` when there is
nothing to add:


In [6]:
names = ["rows", "readings", "days", "sum", "mean", "coldest", "warmest"]
values = conn.execute(f"""
    SELECT COUNT(*), COUNT(r.celsius), COUNT(DISTINCT date(r.hour)),
           ROUND(SUM(r.celsius), 1), ROUND(AVG(r.celsius), 2), MIN(r.celsius), MAX(r.celsius)
    {JOINED}
    WHERE {SVALBARD_IN_MARCH}
""").fetchone()
print("Svalbard in March:", dict(zip(names, values)))

nothing = conn.execute("""
    SELECT COUNT(*), COUNT(celsius), SUM(celsius), TOTAL(celsius), AVG(celsius) FROM readings WHERE hour < '2000'
""")
print("over no rows at all (COUNT(*), COUNT, SUM, TOTAL, AVG):", nothing.fetchone())


Svalbard in March: {'rows': 744, 'readings': 720, 'days': 31, 'sum': -6682.2, 'mean': -9.28, 'coldest': -14.7, 'warmest': -4.0}
over no rows at all (COUNT(*), COUNT, SUM, TOTAL, AVG): (0, 0, None, 0.0, None)


744 rows but 720 readings, since `COUNT(r.celsius)` skipped the 24 missing ones, as `SUM`, `AVG`,
`MIN` and `MAX` did. Over no rows, the counts are 0, `SUM` and `AVG` are `NULL`, and `TOTAL` is 0.0.
The sum and the mean are rounded here because the last digits of a sum of floating point numbers can
differ between versions of SQLite.

With `GROUP BY`, every aggregate works per group, and `HAVING` keeps the groups whose aggregates
pass a condition. `FILTER (WHERE ...)`, which needs SQLite 3.30.0 as `NULLS LAST` does, limits one
aggregate to some of a group's rows. A comparison such as `r.celsius < 0` is 1 or 0, so its average
is the share of readings for which it is true:


In [7]:
for row in conn.execute(f"""
    SELECT s.name,
           COUNT(r.celsius) AS readings,
           COUNT(*) FILTER (WHERE r.celsius < 0) AS frost_hours,
           ROUND(100.0 * AVG(r.celsius < 0), 1) AS frost_percent
    {JOINED}
    GROUP BY s.id
    HAVING AVG(r.celsius < 0) > 0.2
    ORDER BY frost_percent DESC
"""):
    print(row)

print("SQLite's rule:  ", conn.execute(f"SELECT MIN(r.celsius), r.hour {JOINED} WHERE s.name = 'Svalbard'").fetchone())
print("ORDER BY, LIMIT:", conn.execute(f"""
    SELECT r.celsius, r.hour {JOINED} WHERE s.name = 'Svalbard' ORDER BY r.celsius NULLS LAST LIMIT 1
""").fetchone())


('Svalbard', 8736, 5871, 67.2)
('Tromso', 8760, 3203, 36.6)
('Oslo', 8760, 1848, 21.1)
SQLite's rule:   (-17.3, '2025-01-12T03:00')
ORDER BY, LIMIT: (-17.3, '2025-01-12T03:00')


`HAVING` kept the three stations below freezing for more than a fifth of their readings, and dropped
Bergen. The last two queries found the hour of Svalbard's coldest reading. The first relies on a rule
of SQLite's own: in a query with `MIN` or `MAX`, a column that is neither grouped nor aggregated,
`r.hour` here, takes its value from the row that held the minimum or maximum, or from any one of them
when several tie. PostgreSQL refuses such a query. The second sorts and keeps one row, which works in
PostgreSQL too, and needs `NULLS LAST`, or Svalbard's missing readings would sort first.

### Dates, times and timestamps in WHERE

SQLite's date and time functions read a time written as ISO 8601 text, which is how every hour in
this guide is stored, and return a part of it or a new time. `strftime` formats any part,
`julianday` counts days as a real number, and a **modifier** such as `'+1 month'` or
`'start of month'` moves a time before it is returned:


In [8]:
for expression in [
    "date('2025-03-01T06:00')",
    "time('2025-03-01T06:00')",
    "datetime('2025-03-01T06:00')",
    "strftime('%Y-%m', '2025-03-01T06:00')",
    "strftime('%H', '2025-03-01T06:00')",
    "strftime('%w', '2025-03-01T06:00')",
    "julianday('2025-03-01T06:00')",
    "strftime('%s', '2025-03-01T06:00')",
    "datetime(1740808800, 'unixepoch')",
    "date('2025-03-15', 'start of month')",
    "date('2025-03-15', 'start of month', '+1 month', '-1 day')",
    "date('2025-03-15', 'weekday 1')",
    "date('2025-12-31', '-6 days')",
]:
    print(f"{expression:<58} {conn.execute(f'SELECT {expression}').fetchone()[0]!r}")


date('2025-03-01T06:00')                                   '2025-03-01'
time('2025-03-01T06:00')                                   '06:00:00'
datetime('2025-03-01T06:00')                               '2025-03-01 06:00:00'
strftime('%Y-%m', '2025-03-01T06:00')                      '2025-03'
strftime('%H', '2025-03-01T06:00')                         '06'
strftime('%w', '2025-03-01T06:00')                         '6'
julianday('2025-03-01T06:00')                              2460735.75
strftime('%s', '2025-03-01T06:00')                         '1740808800'
datetime(1740808800, 'unixepoch')                          '2025-03-01 06:00:00'
date('2025-03-15', 'start of month')                       '2025-03-01'
date('2025-03-15', 'start of month', '+1 month', '-1 day') '2025-03-31'
date('2025-03-15', 'weekday 1')                            '2025-03-17'
date('2025-12-31', '-6 days')                              '2025-12-25'


`datetime` writes a space where the stored hours have a `T`, which matters when its result is
compared with them. `strftime('%w')` numbers the days of the week from 0 for Sunday, so 1 March 2025
was a Saturday. `strftime('%s')` gives a Unix timestamp, the seconds since the start of 1970 in UTC,
as text, and `'unixepoch'` reads one back. The modifiers found the first of a month, its last day,
the next Monday, and the first day of the year's last week. SQLite reads a time with no zone as UTC,
and the modifier `'localtime'` and the time `'now'` give answers that depend on the machine and the
moment, so this notebook does not print them.

Every one of those works in `WHERE`. A range of text finds a span of time, a function finds a part of
every time, and `julianday` measures between two:


In [9]:
for description, condition in [
    ("Svalbard in March, as a range", SVALBARD_IN_MARCH),
    ("Svalbard on 1 March, with date()", "s.name = 'Svalbard' AND date(r.hour) = '2025-03-01'"),
    ("Svalbard in March, with strftime()", "s.name = 'Svalbard' AND strftime('%Y-%m', r.hour) = '2025-03'"),
    ("Svalbard at 06:00 in March", f"{SVALBARD_IN_MARCH} AND strftime('%H', r.hour) = '06'"),
    ("Oslo on March's weekends", "s.name = 'Oslo' AND r.hour >= '2025-03-01' AND r.hour < '2025-04-01'"
                                 " AND strftime('%w', r.hour) IN ('0', '6')"),
    ("Oslo in the last week of 2025", "s.name = 'Oslo' AND r.hour >= date('2025-12-31', '-6 days')"),
]:
    print(f"{description:<38} {count_where(condition):>4} rows")

between = conn.execute(f"""
    SELECT ROUND((julianday(MAX(r.hour)) - julianday(MIN(r.hour))) * 24, 1) {JOINED} WHERE {SVALBARD_IN_MARCH}
""").fetchone()[0]
print("hours from Svalbard's first reading in March to its last:", between)


Svalbard in March, as a range           744 rows
Svalbard on 1 March, with date()         24 rows
Svalbard in March, with strftime()      744 rows
Svalbard at 06:00 in March               31 rows
Oslo on March's weekends                240 rows
Oslo in the last week of 2025           168 rows
hours from Svalbard's first reading in March to its last: 743.0


March has 744 hours, 1 March 24, and 31 of them at 06:00. March 2025 had five Saturdays and five
Sundays, 240 hours at Oslo, and the last week of the year 168. The hours between two times are the
difference of their Julian day numbers times 24, rounded, since a Julian day is a real number whose
last digits are not exact.

Many systems store a time as a Unix timestamp, a whole number of seconds, and then the range in
`WHERE` has to be numbers too. `CREATE TABLE ... AS SELECT` makes a table from a query, here with
every hour turned into seconds, and `strftime('%s')` turns the edges of March into seconds the same
way. It returns text, so `CAST` makes them integers like the stored timestamps:


In [10]:
conn.execute("""
    CREATE TABLE stamped AS
    SELECT station_id, CAST(strftime('%s', hour) AS INTEGER) AS at, celsius FROM readings
""")
print("the first timestamp:", conn.execute("SELECT at, typeof(at) FROM stamped ORDER BY at LIMIT 1").fetchone())

in_march = conn.execute("""
    SELECT COUNT(*), datetime(MIN(at), 'unixepoch'), datetime(MAX(at), 'unixepoch')
    FROM stamped
    WHERE at >= CAST(strftime('%s', '2025-03-01') AS INTEGER) AND at < CAST(strftime('%s', '2025-04-01') AS INTEGER)
""").fetchone()
conn.execute("DROP TABLE stamped")
print("rows in March, the first and the last:", in_march)


the first timestamp: (1735689600, 'integer')
rows in March, the first and the last: (2976, '2025-03-01 00:00:00', '2025-03-31 23:00:00')


2,976 rows, 744 hours at each of four stations, and `'unixepoch'` turned the first and last back into
times anyone can read. SQLite 3.38.0, from 2022, added `unixepoch('2025-03-01')`, a shorter way to
the same number, and a Python built with an older SQLite does not have it.

### Statements that change data and structure

The other sub-languages at work, on a table made for the purpose: DDL creates, changes and drops it,
and DML fills, updates and deletes its rows. `INSERT` can take its rows from a `SELECT`:


In [11]:
conn.executescript("""
    CREATE TABLE monthly_means (station TEXT NOT NULL, month TEXT NOT NULL, mean REAL);  -- DDL

    INSERT INTO monthly_means (station, month, mean)                                     -- DML
        SELECT s.name, strftime('%Y-%m', r.hour), ROUND(AVG(r.celsius), 1)
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE r.hour < '2025-04-01'
        GROUP BY s.id, strftime('%Y-%m', r.hour);

    UPDATE monthly_means SET mean = NULL WHERE station = 'Svalbard' AND month = '2025-03';  -- DML
    DELETE FROM monthly_means WHERE month = '2025-01';                                       -- DML
    ALTER TABLE monthly_means ADD COLUMN note TEXT;                                          -- DDL
""")

print("rows and months left:", conn.execute("SELECT COUNT(*), COUNT(DISTINCT month) FROM monthly_means").fetchone())
print("columns:", [column[1] for column in conn.execute("PRAGMA table_info(monthly_means)")])
svalbard = "SELECT month, mean FROM monthly_means WHERE station = 'Svalbard' ORDER BY month"
print("Svalbard:", conn.execute(svalbard).fetchall())

conn.execute("DROP TABLE monthly_means")                                                    # DDL
left = conn.execute("SELECT COUNT(*) FROM sqlite_schema WHERE name = 'monthly_means'").fetchone()[0]
print("tables called monthly_means after DROP TABLE:", left)


rows and months left: (8, 2)
columns: ['station', 'month', 'mean', 'note']
Svalbard: [('2025-02', -12.4), ('2025-03', None)]
tables called monthly_means after DROP TABLE: 0


The `INSERT` added twelve rows, four stations by three months, the `UPDATE` set one mean to `NULL`,
and the `DELETE` removed January's four. `ALTER TABLE` added a column, one of the few changes
SQLite's `ALTER TABLE` can make, along with renaming a table or a column and dropping a column, and
the **Changing a Schema** notebook covers the rest. `DROP TABLE` removed the table and its rows. A
comment in Python code, after `#`, is Python's, and one inside the SQL, after `--`, is SQL's.

### Comments, semicolons and case

Keywords in any case, names in any case, comments of both kinds, and a closing semicolon, in a
statement `execute` runs as it would any other:


In [12]:
query = """
    select name, latitude            -- keywords in lower case work the same
    from STATIONS                    /* and names match whatever their case */
    where Latitude > 69
    order by LATITUDE desc;
"""
print(conn.execute(query).fetchall())


[('Svalbard', 78.22), ('Kirkenes', 69.73), ('Tromso', 69.65)]


The capitals in this guide's SQL are a convention for the reader, and SQLite reads the query above
the same way. A statement needs no semicolon in `execute`, and a script in `executescript` needs one
between every two statements, as the **Tables and Queries** notebook showed.

### A range, date(), strftime() or BETWEEN

A time in `WHERE` can be tested four ways, and they do not all mean the same thing:

| Write | When | Why |
|---|---|---|
| `hour >= '2025-03-01' AND hour < '2025-04-01'` | any span of time: a day, a month, a year | it compares the stored text directly, so an index on the column can serve it, and it takes in every time in the span |
| `date(hour) = '2025-03-01'` | one day, in a query that reads few rows anyway | it reads most plainly, but the function runs on every row, so an index on `hour` cannot help |
| `strftime('%H', hour) = '06'` | a part of every time, such as an hour of the day, a month or a weekday | no range can say every 06:00 of the year, so only a function can |
| `hour BETWEEN '2025-03-01' AND '2025-03-31'` | values that hold a date and no time | a time on the last day sorts after the date alone, so the last day is left out, which a Common error shows |

The default for a span is the range, with `>=` at the start and `<` at the first moment after it.

### A monthly frost summary

The pieces of this notebook in one query: when frost set in at the two Arctic stations. For Tromso
and Svalbard in the second half of 2025, it lists the months in which most readings were below
freezing, with every month's readings, mean, coldest reading, share of frost and number of days with
any frost, in order. A range of text chooses the half year, `strftime` makes the months, `HAVING`
keeps the frosty ones, and `FILTER` and `COUNT(DISTINCT ...)` count the frost days:


In [13]:
summary = f"""
    SELECT s.name AS station,
           strftime('%Y-%m', r.hour) AS month,
           COUNT(r.celsius) AS readings,
           ROUND(AVG(r.celsius), 1) AS mean,
           MIN(r.celsius) AS coldest,
           ROUND(100.0 * AVG(r.celsius < 0), 1) AS frost_percent,
           COUNT(DISTINCT date(r.hour)) FILTER (WHERE r.celsius < 0) AS frost_days
    {JOINED}
    WHERE s.name IN ('Tromso', 'Svalbard')
      AND r.hour >= '2025-07-01' AND r.hour < '2026-01-01'
    GROUP BY s.id, strftime('%Y-%m', r.hour)
    HAVING AVG(r.celsius < 0) > 0.5
    ORDER BY station, month
"""
print(f"{'station':<9} {'month':<8} {'readings':>8} {'mean':>6} {'coldest':>8} {'frost %':>8} {'frost days':>11}")
for station, month, readings, mean, coldest, frost_percent, frost_days in conn.execute(summary):
    print(f"{station:<9} {month:<8} {readings:>8} {mean:>6} {coldest:>8} {frost_percent:>8} {frost_days:>11}")


station   month    readings   mean  coldest  frost %  frost days
Svalbard  2025-10       744   -4.2     -9.8     94.8          31
Svalbard  2025-11       720   -8.7    -13.9    100.0          30
Svalbard  2025-12       744  -12.1    -16.8    100.0          31
Tromso    2025-11       720   -0.7     -6.0     58.1          30
Tromso    2025-12       744   -4.1     -8.9     97.8          31


Most of Svalbard's readings were below freezing from October, and most of Tromso's from November.
September at Svalbard, at 47.4 percent, and October at Tromso, at 7.3, fell short of `HAVING`, which
tested every group's share before `SELECT` rounded it. `GROUP BY` names two expressions, so every
station and month is a group of its own.

### Where each part came from

| In the summary | What it relies on | The section that showed it |
|---|---|---|
| `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `HAVING` and `ORDER BY`, in that order | the order SQL requires, and the order the clauses take effect | The clauses, in the order they take effect |
| `s.name IN ('Tromso', 'Svalbard') AND r.hour >= '2025-07-01' AND r.hour < '2026-01-01'` | a list, conditions joined by `AND`, and a range of time | WHERE: conditions, and how AND and OR combine |
| `strftime('%Y-%m', r.hour)` in `SELECT` and `GROUP BY` | a part of every time, worked out by a function | Dates, times and timestamps in WHERE |
| `COUNT(r.celsius)`, `AVG(r.celsius)` and `MIN(r.celsius)` | aggregates over every group, skipping `NULL` | Aggregate functions |
| `ROUND(100.0 * AVG(r.celsius < 0), 1)` | the share of readings a comparison is true for | Aggregate functions |
| `COUNT(DISTINCT date(r.hour)) FILTER (WHERE r.celsius < 0)` | an aggregate over part of a group, counting different values | Aggregate functions |
| `HAVING AVG(r.celsius < 0) > 0.5` | a condition on a group, after aggregating | The clauses, in the order they take effect |
| `ORDER BY station, month` | a sort by one expression, then the next | ORDER BY and LIMIT |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/04-sql-syntax-solutions.ipynb).

**1.** Print the three days in January with Svalbard's lowest mean temperature, coldest first, with
each day's mean to one decimal place.


In [14]:
# your code here


**2.** For every station, Kirkenes included, print the number of readings, the number of missing
readings, and the share of its readings below freezing as a percentage to one decimal place.


In [15]:
# your code here


**3.** Count Tromso's readings on the weekends of February 2025, using `strftime`.


In [16]:
# your code here


**4.** Print how many hours passed between Svalbard's coldest reading of 2025 and its warmest, using
`julianday`.


In [17]:
# your code here


**5.** Print the months in which Bergen had more than 100 hours below freezing, with the number of
hours, using `HAVING`.


In [18]:
# your code here


**6.** Create a table `monthly_coldest` with a station, a month and a coldest reading. Fill it with
one `INSERT ... SELECT` for every station and month of 2025, set Oslo's coldest reading for December
to `NULL` with `UPDATE`, delete every row with no coldest reading with `DELETE`, print how many rows
are left, and drop the table.


In [19]:
# your code here


## Common errors

### sqlite3.OperationalError: near "3": syntax error


In [20]:
conn.execute("SELECT TOP 3 hour, celsius FROM readings ORDER BY celsius DESC")


OperationalError: near "3": syntax error

`TOP` is SQL Server's way to limit rows, written straight after `SELECT`. SQLite has no `TOP`, so it
read the word as the name of a column, and then could make no sense of the 3 after it, which is why
the message names the 3 and not `TOP`. SQLite writes the limit at the end, as `LIMIT`, and here
`hour` decides between readings that tie at 20.6 degrees:


In [21]:
print(conn.execute("SELECT hour, celsius FROM readings ORDER BY celsius DESC, hour LIMIT 3").fetchall())


[('2025-07-21T15:00', 20.8), ('2025-07-16T16:00', 20.7), ('2025-07-08T15:00', 20.6)]


### sqlite3.OperationalError: near "WHERE": syntax error


In [22]:
conn.execute(f"""
    SELECT s.name, COUNT(*)
    {JOINED}
    GROUP BY s.id
    WHERE r.celsius < 0
""")


OperationalError: near "WHERE": syntax error

SQL requires `WHERE` before `GROUP BY`, and SQLite read a `WHERE` after it as a mistake. The written
order is fixed, whatever order the clauses take effect in. A condition on single rows goes in
`WHERE`, before `GROUP BY`, and a condition on groups goes in `HAVING`, after it:


In [23]:
print(conn.execute(f"""
    SELECT s.name, COUNT(*)
    {JOINED}
    WHERE r.celsius < 0
    GROUP BY s.id
    ORDER BY s.name
""").fetchall())


[('Bergen', 1220), ('Oslo', 1848), ('Svalbard', 5871), ('Tromso', 3203)]


### sqlite3.OperationalError: near "TRUNCATE": syntax error


In [24]:
conn.execute("CREATE TABLE draft (note TEXT)")
conn.execute("INSERT INTO draft VALUES ('first'), ('second')")
conn.execute("TRUNCATE TABLE draft")


OperationalError: near "TRUNCATE": syntax error

`TRUNCATE TABLE` belongs to other databases' DDL, and SQLite does not have it, or `GRANT` and
`REVOKE` either, since it has no users to grant anything to. `DELETE` with no `WHERE` empties a
table and keeps it, and on a table with no triggers SQLite does that all at once, not one row at a
time:


In [25]:
conn.execute("DELETE FROM draft")
print("rows left:", conn.execute("SELECT COUNT(*) FROM draft").fetchone()[0])
conn.execute("DROP TABLE draft")
conn.commit()


rows left: 0


### No error, and the last day of March missing: BETWEEN with times


In [26]:
in_march = conn.execute(f"""
    SELECT COUNT(*) {JOINED}
    WHERE s.name = 'Svalbard' AND r.hour BETWEEN '2025-03-01' AND '2025-03-31'
""").fetchone()[0]
print("Svalbard's hours in March:", in_march)


Svalbard's hours in March: 720


March has 744 hours, and the query counted 720. `BETWEEN` compared text, and `'2025-03-31T00:00'` is
greater than `'2025-03-31'`, since a longer text that starts the same sorts after the shorter one, so
every hour of 31 March was outside the range. The range that includes all of a span ends at the first
moment after it, with `<`:


In [27]:
in_march = conn.execute(f"SELECT COUNT(*) {JOINED} WHERE {SVALBARD_IN_MARCH}").fetchone()[0]
print("Svalbard's hours in March:", in_march)


Svalbard's hours in March: 744


### No error, and warm hours counted as frost: AND beside OR


In [28]:
frost_hours = count_where("s.name = 'Oslo' OR s.name = 'Bergen' AND r.celsius < 0")
print("hours below freezing at Oslo or Bergen:", frost_hours)


hours below freezing at Oslo or Bergen: 9980


Oslo and Bergen together had 3,068 hours below freezing, and the query counted 9,980. `AND` binds
more tightly than `OR`, so the condition meant every Oslo reading, or a Bergen reading below
freezing, and counted all 8,760 of Oslo's hours. Parentheses put the stations together first:


In [29]:
frost_hours = count_where("(s.name = 'Oslo' OR s.name = 'Bergen') AND r.celsius < 0")
print("hours below freezing at Oslo or Bergen:", frost_hours)


hours below freezing at Oslo or Bergen: 3068


### No error, and 0 percent: dividing two counts


In [30]:
print(conn.execute(f"""
    SELECT s.name, COUNT(*) FILTER (WHERE r.celsius < 0) / COUNT(r.celsius) * 100 AS frost_percent
    {JOINED}
    GROUP BY s.id
    ORDER BY s.name
""").fetchall())


[('Bergen', 0), ('Oslo', 0), ('Svalbard', 0), ('Tromso', 0)]


Every station's share came out as 0. Both counts are integers, and SQLite divides two integers as
integers, dropping the fraction, so Bergen's 1,220 readings below freezing out of 8,760 were 0
before the multiplication by 100 began. A real number anywhere in the arithmetic makes the division
keep its fraction, and putting `100.0` first does it before the division:


In [31]:
print(conn.execute(f"""
    SELECT s.name, ROUND(100.0 * COUNT(*) FILTER (WHERE r.celsius < 0) / COUNT(r.celsius), 1) AS frost_percent
    {JOINED}
    GROUP BY s.id
    ORDER BY s.name
""").fetchall())


[('Bergen', 13.9), ('Oslo', 21.1), ('Svalbard', 67.2), ('Tromso', 36.6)]


### No error, and January after February: dates written day first


In [32]:
conn.execute("CREATE TABLE visits (station TEXT NOT NULL, day TEXT NOT NULL)")
conn.executemany("INSERT INTO visits VALUES (?, ?)",
                 [("Oslo", "31.01.2025"), ("Oslo", "03.02.2025"), ("Oslo", "15.12.2024")])   # as Norwegian forms write dates

print("in order:  ", conn.execute("SELECT day FROM visits ORDER BY day").fetchall())
print("in 2025:   ", conn.execute("SELECT day FROM visits WHERE day >= '01.01.2025'").fetchall())


in order:   [('03.02.2025',), ('15.12.2024',), ('31.01.2025',)]
in 2025:    [('31.01.2025',), ('03.02.2025',), ('15.12.2024',)]


The visits sorted with 3 February first and 31 January last, and all three counted as 2025,
including the one in December 2024. Text compares from its first character, and a date written day
first puts the day there, so the text sorts by day of the month. Only ISO 8601's order, year, month,
then day, sorts as time does, and SQLite's date functions read only that order too. Convert the
stored text once, with `substr`, and keep it in that form:


In [33]:
conn.execute("UPDATE visits SET day = substr(day, 7, 4) || '-' || substr(day, 4, 2) || '-' || substr(day, 1, 2)")
conn.commit()

print("in order:  ", conn.execute("SELECT day FROM visits ORDER BY day").fetchall())
print("in 2025:   ", conn.execute("SELECT day FROM visits WHERE day >= '2025-01-01'").fetchall())
conn.close()


in order:   [('2024-12-15',), ('2025-01-31',), ('2025-02-03',)]
in 2025:    [('2025-01-31',), ('2025-02-03',)]


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [34]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A `SELECT` is written `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`, `LIMIT`, and
  takes effect `FROM`, `WHERE`, `GROUP BY`, `HAVING`, `SELECT`, `ORDER BY`, `LIMIT`.
- SQL's statements query data, change rows, define structures, and control access, and SQLite has no
  `TRUNCATE`, `GRANT`, `REVOKE` or `TOP`.
- Keywords and names ignore case, `--` and `/* */` are comments, and a semicolon ends a statement.
- `AND` binds more tightly than `OR`, so a mixed condition needs parentheses.
- `COUNT(*)` counts rows, the other aggregates skip `NULL`, `HAVING` and `FILTER` narrow them, and
  dividing two integers drops the fraction unless a real number such as `100.0` is in the arithmetic.
- A span of time in `WHERE` is a range, `>=` its start and `<` the moment after it, never `BETWEEN`
  two dates for values that hold times.
- `date`, `time`, `datetime`, `strftime` and `julianday`, with modifiers, work out the parts of a
  time and the hours between two, and dates stored as ISO 8601 text sort and compare as time does.
- A column of Unix timestamps is compared with numbers, which `CAST` makes from `strftime('%s')`.


## What is next

The **Joins** notebook takes the `FROM` clause apart: inner, left, right, full, cross and self
joins, each shown on the same two small tables, with what every one of them does with a row that has
no match.


---

&#8592; **Previous:** [Tables and Queries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/03-tables-and-queries.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
